# Preprocessing

This notebook documents the project pipeline, supported raw sources, and stored processed outputs under the academic folder structure.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dataset").exists() and (candidate / "src").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import DATASET_SPECS

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed_data"
RAW_DIR = PROJECT_ROOT / "dataset" / "raw_data"
CATALOG_PATH = PROCESSED_DIR / "dataset_catalog.json"

with CATALOG_PATH.open("r", encoding="utf-8") as handle:
    catalog = json.load(handle)

pd.DataFrame(catalog["datasets"])[["title", "source_file", "processed_file", "processed_rows"]]


In [ ]:
pd.DataFrame(
    {
        "dataset_name": [spec.name for spec in DATASET_SPECS],
        "source_file": [spec.source_file for spec in DATASET_SPECS],
        "processed_file": [spec.processed_file for spec in DATASET_SPECS],
        "chunked_reader": ["yes" if spec.chunk_size else "no" for spec in DATASET_SPECS],
    }
)


In [ ]:
pd.concat(
    [
        pd.Series(sorted(path.name for path in RAW_DIR.glob("*.csv")), name="raw_files_present"),
        pd.Series(sorted(path.name for path in PROCESSED_DIR.glob("*_processed.csv")), name="processed_outputs_present"),
    ],
    axis=1,
)


In [ ]:
summary_assets = sorted(
    path.name for path in PROCESSED_DIR.glob("*")
    if path.is_file() and not path.name.endswith("_processed.csv")
)
pd.DataFrame({"stored_assets": summary_assets})
